# Agentic AI systems: reading trajectories

Exploration only. Run `make agent-eval` first so `artifacts/traces` and `artifacts/agent_eval.json` exist.

## Every run, at a glance

In [1]:
import json
from pathlib import Path
from nw.agent.trace import Trajectory, replay
traces = [Trajectory.load(p) for p in sorted(Path("../artifacts/traces").glob("*.json"))]
print(len(traces), "runs")
for t in traces[:20]:
    print(f"{t.run_id}  {t.agent:12s} {t.terminated.value:9s} steps={t.n_steps:2d} cost={t.cost_usd:.4f}  tools={t.tools_called}")

65 runs
0a722c9c1d  resolver     answer    steps= 2 cost=0.0016  tools=['search_policies']
0d56d65396  resolver     answer    steps= 8 cost=0.1443  tools=['lookup_customer', 'check_entitlement', 'classify_urgency', 'classify_semantic', 'find_similar_tickets', 'search_policies', 'search_policies']
0e170ec7ce  resolver     answer    steps= 4 cost=0.0642  tools=['lookup_customer', 'check_entitlement', 'search_policies']
21b2c23139  resolver     answer    steps= 4 cost=0.0818  tools=['lookup_customer', 'check_entitlement', 'search_policies']
270e708ecf  resolver     answer    steps= 2 cost=0.0016  tools=['lookup_customer']
27836a89d2  resolver     answer    steps= 3 cost=0.0853  tools=['search_policies', 'lookup_customer']
2b40f216e5  resolver     answer    steps= 2 cost=0.0016  tools=['search_policies']
2cb534b35b  resolver     answer    steps= 7 cost=0.1997  tools=['lookup_customer', 'classify_urgency', 'classify_semantic', 'find_similar_tickets', 'search_policies', 'escalate']
30b3512cd

## The failures, and the step where each went wrong

In [2]:
ev = json.load(open("../artifacts/agent_eval.json"))
failed = [s for s in ev["scores"] if not s["success"]]
print(len(failed), "failed of", ev["aggregate"]["n"])
for s in failed:
    print("\n==", s["id"], s["kind"], s["failures"])
    t = Trajectory.load(Path("../artifacts/traces") / f"{s['trace']}.json")
    print(replay(t)[:1500])

0 failed of 35


## Cost per resolution by kind

In [3]:
import collections, statistics
by_kind = collections.defaultdict(list)
for s in ev["scores"]:
    by_kind[s["kind"]].append(s["cost_usd"])
for k, v in sorted(by_kind.items(), key=lambda kv: -statistics.mean(kv[1])):
    print(f"{k:16s} n={len(v)} mean {statistics.mean(v):.4f} USD max {max(v):.4f}")
print("overall per resolution:", round(ev["aggregate"]["cost_usd_per_resolution"], 4), "USD")

p0               n=1 mean 0.0024 USD max 0.0024
security         n=2 mean 0.0024 USD max 0.0024
p0_outage        n=1 mean 0.0024 USD max 0.0024
p0_data_loss     n=1 mean 0.0024 USD max 0.0024
refund_over_limit n=1 mean 0.0024 USD max 0.0024
german_p0        n=1 mean 0.0024 USD max 0.0024
injection        n=2 mean 0.0016 USD max 0.0016
unanswerable     n=1 mean 0.0016 USD max 0.0016
contradiction    n=1 mean 0.0016 USD max 0.0016
entitlement      n=3 mean 0.0016 USD max 0.0016
sequence         n=1 mean 0.0016 USD max 0.0016
refund_limit     n=1 mean 0.0016 USD max 0.0016
injection_tool   n=1 mean 0.0016 USD max 0.0016
pii              n=1 mean 0.0016 USD max 0.0016
german           n=1 mean 0.0016 USD max 0.0016
internal_leak    n=1 mean 0.0016 USD max 0.0016
rate_limit       n=1 mean 0.0016 USD max 0.0016
cross_account    n=1 mean 0.0016 USD max 0.0016
similar_leak     n=1 mean 0.0016 USD max 0.0016
howto            n=4 mean 0.0016 USD max 0.0016
entitlement_missing n=1 mean 0.0016 USD

## Questions to bring back

- For each failure: tool design, prompt, or the case itself?
- What would the cost per resolution be at Northwind's 8,000 tickets a quarter, and who signs off on that?